In [1]:
import numpy as np
from numpy import sin, pi, cos

# 1. Maintain your exact spatial configuration
nx = 600
ny = 1200
taux_max = 0.2

# 2. Define the high-frequency temporal parameters
dt_forcing = 3600.0         # 1-hour intervals (in seconds)
total_days = 40             # Keep file size reasonable (approx. 4.1 GB for 40 days)
total_seconds = total_days * 86400
time_steps = int(total_seconds / dt_forcing)

# Setup math frequencies (Example: ~17.5 hour inertial + 4 day storm cycles)
t_inertial = 62832.0       
t_storm = 345600.0          
omega_i = 2 * pi / t_inertial
omega_s = 2 * pi / t_storm

# 3. Initialize the 3D array: Shape must be (time, ny, nx)
wind_3d = np.zeros([time_steps, ny, nx])

# 4. Loop through time to build your profile dynamically
for t_idx in range(time_steps):
    current_time = t_idx * dt_forcing
    
    # Calculate temporal multipliers
    high_freq = cos(omega_i * current_time)
    storm_envelope = 1.0 + 0.5 * cos(omega_s * current_time)
    
    # Generate your exact baseline spatial profile
    spatial_sin = sin(np.linspace(0, pi, 1190))
    
    # Scale it by the temporal function
    dynamic_spatial_profile = taux_max * spatial_sin * high_freq * storm_envelope
    
    # Inject it into the time frame, keeping your [5:1195] wall buffers zeroed
    wind_3d[t_idx, 5:1195, :] = np.reshape(dynamic_spatial_profile, [1190, 1])

# 5. Output using your exact float32 (single precision) big-endian format
wind_3d.astype('>f4').tofile('zonal_wind_time_varying.5km.bin')

print(f"File generated successfully! Array shape: {wind_3d.shape}")

File generated successfully! Array shape: (960, 1200, 600)
